# 3. Checkpoint, memoria lunga e compaction

Questo notebook mostra tre livelli distinti dell'ecosistema LangChain/LangGraph:

- **short-term memory**: messaggi salvati in un thread tramite checkpointer SQLite;
- **compaction**: sintesi automatica della cronologia tramite middleware;
- **long-term memory**: informazioni condivise tra thread tramite LangGraph Store.

Gli esempi usano un modello OpenAI reale e non dipendono da file Python esterni.

## Configurazione

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


def find_env() -> Path:
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        candidate = directory / '.env'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('.env non trovato.')


ENV_FILE = find_env()
load_dotenv(ENV_FILE, override=False)
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError(f'OPENAI_API_KEY non configurata in {ENV_FILE}')

MODEL_NAME = os.getenv('OPENAI_MODEL') or 'gpt-5.4-mini'
model = ChatOpenAI(
    model=MODEL_NAME,
    reasoning_effort='low',
    use_responses_api=True,
    store=False,
)
print('Modello:', MODEL_NAME)

## Short-term memory persistente e compaction

`SqliteSaver` salva lo stato dopo ogni avanzamento del graph. Riutilizzando `thread_id`, ogni nuova invocazione recupera automaticamente la cronologia. `SummarizationMiddleware` entra in azione prima che il contesto cresca senza controllo.

La soglia di quattro messaggi è volutamente minuscola per rendere osservabile il fenomeno.

In [ ]:
import tempfile

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.sqlite import SqliteSaver

state_directory = tempfile.TemporaryDirectory(prefix='langgraph-state-')
database_path = Path(state_directory.name) / 'checkpoints.sqlite'
thread_config = {'configurable': {'thread_id': 'lezione-compaction'}}

with SqliteSaver.from_conn_string(str(database_path)) as checkpointer:
    checkpointer.setup()
    compact_agent = create_agent(
        model=model,
        tools=[],
        checkpointer=checkpointer,
        middleware=[
            SummarizationMiddleware(
                model=model,
                trigger=('messages', 4),
                keep=('messages', 2),
            )
        ],
        system_prompt=(
            'Ricorda accuratamente codici e fatti dichiarati dall\'utente. '
            'Rispondi in una frase.'
        ),
    )

    first = compact_agent.invoke(
        {'messages': [{'role': 'user', 'content': 'Il codice del progetto è ZETA-42.'}]},
        config=thread_config,
    )
    second = compact_agent.invoke(
        {'messages': [{'role': 'user', 'content': 'Il colore del progetto è blu.'}]},
        config=thread_config,
    )
    third = compact_agent.invoke(
        {'messages': [{'role': 'user', 'content': 'Qual è il codice del progetto?'}]},
        config=thread_config,
    )
    snapshot = compact_agent.get_state(thread_config)

print(third['messages'][-1].text)
print('Messaggi conservati nello snapshot:', len(snapshot.values['messages']))
print('Tipi:', [type(message).__name__ for message in snapshot.values['messages']])
assert 'ZETA-42' in third['messages'][-1].text

### Che cosa è successo?

Al terzo turno la soglia è stata raggiunta. Il middleware ha chiesto al modello una sintesi della parte più vecchia, ha mantenuto la coda recente e ha proseguito. Il checkpoint contiene lo stato risultante, non una lista Python gestita manualmente.

## Long-term memory con LangGraph Store

Un checkpointer separa i thread. Uno Store permette invece di organizzare documenti per namespace e chiave, rendendoli disponibili anche a conversazioni differenti. I tool accedono allo store attraverso `ToolRuntime`.

In [ ]:
from dataclasses import dataclass

from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore


@dataclass
class UserContext:
    user_id: str


@tool
def save_preference(preference: str, runtime: ToolRuntime[UserContext]) -> str:
    '''Salva una preferenza durevole dell'utente nello store.'''
    namespace = ('preferences', runtime.context.user_id)
    runtime.store.put(namespace, 'profile', {'preference': preference})
    return 'Preferenza salvata.'


@tool
def read_preference(runtime: ToolRuntime[UserContext]) -> str:
    '''Legge la preferenza durevole dell'utente dallo store.'''
    namespace = ('preferences', runtime.context.user_id)
    item = runtime.store.get(namespace, 'profile')
    return item.value['preference'] if item else 'Nessuna preferenza salvata.'


store = InMemoryStore()
memory_agent = create_agent(
    model=model,
    tools=[save_preference, read_preference],
    store=store,
    checkpointer=InMemorySaver(),
    context_schema=UserContext,
    system_prompt=(
        'Quando l\'utente chiede di ricordare una preferenza usa save_preference. '
        'Quando chiede cosa preferisce usa read_preference. Non inventare valori.'
    ),
)

memory_agent.invoke(
    {'messages': [{'role': 'user', 'content': 'Ricorda che preferisco risposte con esempi Python.'}]},
    config={'configurable': {'thread_id': 'thread-a'}},
    context=UserContext(user_id='student-1'),
)

remembered = memory_agent.invoke(
    {'messages': [{'role': 'user', 'content': 'Qual è la mia preferenza?'}]},
    config={'configurable': {'thread_id': 'thread-b'}},
    context=UserContext(user_id='student-1'),
)
print(remembered['messages'][-1].text)
assert 'Python' in remembered['messages'][-1].text

## Ispezionare direttamente lo Store

La lettura diretta serve per capire la struttura; nell'agente l'accesso avviene tramite tool e runtime.

In [ ]:
stored_item = store.get(('preferences', 'student-1'), 'profile')
stored_item.value

## Confronto finale

| Meccanismo | Ambito | Scopo |
|---|---|---|
| Checkpointer | singolo thread | ripresa, fault tolerance, HITL |
| Summarization middleware | finestra del modello | riduzione del contesto |
| Store | più thread / utente | memoria semantica durevole |

`InMemoryStore` è sufficiente per la lezione; in produzione va sostituito con uno store persistente e isolato per utente.

In [ ]:
state_directory.cleanup()
print('Database temporaneo eliminato.')